# מפגש 7 — הכנת נתונים למודל KNN
**יחידה 3 | הדגמה מודרכת | מאגר Penguins**

נלמד לזהות ערכים חסרים, לבחור מאפיינים ומטרה, לחלק לאימון ובדיקה, לטפל בחסרים ולבצע MinMax/StandardScaler ללא דליפת מידע. לאחר מכן נבדוק סיווג. הידע על Pipeline הוא העשרה קצרה.

## 1. מכירים את הנתונים
מה רוצים לחזות? נבחר `species`. ארבע מדידות גוף יהיו המאפיינים. `island` ו־`sex` לא יהיו בקבוצת המאפיינים בדוגמת הבסיס.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
display(df.head())
print("גודל הטבלה:", df.shape)
print("מספר חסרים בכל עמודה:")
print(df.isna().sum())

## 2. בוחרים X ו־y
`X` היא טבלת המאפיינים, `y` היא תגית הסיווג. אין לכלול את `species` בין המאפיינים כשזו המטרה.

In [ ]:
features=["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
X=df[features]
y=df["species"]
print("X:",X.shape,"y:",y.shape)

## 3. קודם מחלקים Train/Test
**כלל חשוב:** את ערכי ההשלמה ואת סקלת התקנון לומדים מתוך **Train בלבד**, ומיישמים לאחר מכן על Test. כך לא משתמשים במידע מהבדיקה במהלך הכנת המודל.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Train:",X_train.shape, "Test:",X_test.shape)

## 4. משלימים חסרים על בסיס Train
בחרנו למלא ערכים מספריים חסרים באמצעות חציון. זו בחירה דידקטית אחת; המצגת מציגה גם מחיקת שורות/עמודות והשלמה בממוצע.

In [ ]:
imputer=SimpleImputer(strategy="median")
X_train_filled=imputer.fit_transform(X_train)
X_test_filled=imputer.transform(X_test)
print("ערכי חציון שנלמדו מ-Train:",imputer.statistics_)

## 5. נרמול Min-Max
נחשב את המינימום והמקסימום של Train בלבד. KNN מבוסס מרחק ולכן קנה המידה של מאפיינים עשוי להשפיע על התוצאה.

In [ ]:
scaler=MinMaxScaler()
X_train_scaled=scaler.fit_transform(X_train_filled)
X_test_scaled=scaler.transform(X_test_filled)
print("דוגמה לנרמול:",X_train_scaled[0])

## 6. מתאמנים ובודקים
בבדיקה נשתמש באותן דוגמאות Test, ללא אימון על תוויות הבדיקה.

In [ ]:
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled,y_train)
y_pred=knn.predict(X_test_scaled)
print("Accuracy ב-Test:",round(accuracy_score(y_test,y_pred),3))

## 7. תקנון StandardScaler — להשוואה
בניגוד ל־Min-Max, תקנון משתמש בממוצע ובסטיית תקן של נתוני האימון. השוו את התוצאות, אך אל תבחרו את המודל הסופי לפי ציוני Test חוזרים.

In [ ]:
standard=StandardScaler()
X_train_std=standard.fit_transform(X_train_filled)
X_test_std=standard.transform(X_test_filled)
knn_std=KNeighborsClassifier(n_neighbors=5)
knn_std.fit(X_train_std,y_train)
print("StandardScaler — Test Accuracy:",round(accuracy_score(y_test,knn_std.predict(X_test_std)),3))

## בדיקת הבנה וסיכום
1. במה שונות השלמה, נרמול ותקנון?
2. למה מחלקים נתונים לפני `fit` של ה־Scaler וה־Imputer?
3. למה `species` אינה מופיעה ב־X?
4. האם Accuracy אחד מספיק להשוואת מודלים?
5. כתבו החלטה מקצועית אחת שקיבלתם במהלך הכנת הנתונים ונמקו.